In [1]:
# Это нужно, чтоб увидеть NEURON
import sys
sys.path.append("/home/leon/nrn/build/lib/python")

import os
os.environ["NAVIS_SKIP_CMTK"] = "1"

os.environ["PATH"] = ":".join(
    p for p in os.environ["PATH"].split(":")
    if p != "/root/.cargo/bin"
)

import json
import numpy as np
import pandas as pd
import pickle
import pymaid

os.chdir("../../..")

from scripts.NEURON_Sim_Wrapper import Network
from scripts.Sim_Visializations import *

# Get ALL 3k Neurons and Params

In [2]:
catmaid_url = 'https://l1em.catmaid.virtualflybrain.org'
http_user = None
http_password = None
project_id = 1

rm = pymaid.CatmaidInstance(catmaid_url, http_user, http_password, project_id)

neuron_list = pymaid.get_skids_by_annotation('mw brain and inputs')
neuron_list = list(map(str, neuron_list))

len(neuron_list)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


3016

#### Get the neurite params

In [3]:
with open("./Datasets/Generated/Optimized_Neural_Params/Optimized_Neural_Params(70).json", "r") as f:
    data = json.load(f)

rows = []
for neuron_id, content in data["results"].items():
    p = content["best_params"]
    rows.append({
        "neuron_id": int(neuron_id),
        "gnabar_hh": p["dend_gnabar_hh"],
        "gkbar_hh": p["dend_gkbar_hh"],
        "gl_hh": p["dend_gl_hh"]
    })

neurites_params = pd.DataFrame(rows)

# Cast to string
neurites_params["neuron_id"] = neurites_params["neuron_id"].astype(str)

# Get needed neurons
neurites_params = neurites_params[neurites_params["neuron_id"].isin(neuron_list)]

# Set the rest of params manually
neurites_params['L'] = 5.0
neurites_params['diam'] = 1.0
neurites_params['Ra'] = 100.0
neurites_params['cm'] = 1.0
neurites_params['el_hh'] = -70.0

neurites_params

,neuron_id,gnabar_hh,gkbar_hh,gl_hh,L,diam,Ra,cm,el_hh
0,11192283,0.023904,0.016436,0.000232,5.0,1.0,100.0,1.0,-70.0
1,3771794,0.071014,0.135597,0.000244,5.0,1.0,100.0,1.0,-70.0
2,19470184,0.024574,0.027542,0.000087,5.0,1.0,100.0,1.0,-70.0
4,3639968,0.017545,0.005276,0.000273,5.0,1.0,100.0,1.0,-70.0
6,9578602,0.030087,0.022530,0.000315,5.0,1.0,100.0,1.0,-70.0
...,...,...,...,...,...,...,...,...,...
3790,8864096,0.032983,0.011746,0.000525,5.0,1.0,100.0,1.0,-70.0
3793,8689674,0.021775,0.016568,0.000118,5.0,1.0,100.0,1.0,-70.0
3797,12364717,0.025734,0.005005,0.000508,5.0,1.0,100.0,1.0,-70.0
3798,16339338,0.021675,0.005425,0.000175,5.0,1.0,100.0,1.0,-70.0


#### Get Syn Types

In [4]:
# Типы синапсов по нейронам: из Syn_Types(By_Cell_Types).csv
# excitatory -> exc, inhibitory -> inh; для mixed/пустых/прочих — 70% exc / 30% inh (фиксированный seed)
import numpy as np

SYN_TYPES_CSV = "./Datasets/Generated/Syn_Types(By_Cell_Types).csv"
RANDOM_OTHER_EXC_PROB = 0.8
RANDOM_SEED = 42

syn_types_df = pd.read_csv(SYN_TYPES_CSV)
syn_types_df["neuron_id"] = syn_types_df["neuron_id"].astype(str)

# только нейроны из нашей сети
syn_types_df = syn_types_df[syn_types_df["neuron_id"].isin(neuron_list)].drop_duplicates(subset="neuron_id", keep="first")

stype_map = {}
rng = np.random.default_rng(RANDOM_SEED)
for _, row in syn_types_df.iterrows():
    nid = row["neuron_id"]
    val = row["synapse_type"]
    raw = "" if pd.isna(val) else str(val).strip().lower()
    
    if raw == "excitatory":
        stype_map[nid] = "exc"
    elif raw == "inhibitory":
        stype_map[nid] = "inh"
    else:
        # mixed, пустое или любое другое — рандом
        stype_map[nid] = "exc" if rng.random() < RANDOM_OTHER_EXC_PROB else "inh"

# нейроны без записи в CSV по умолчанию считаем возбуждающими (обёртка так же использует exc при отсутствии в syn_types)
for nid in neuron_list:
    if nid not in stype_map:
        stype_map[nid] = "exc"
        
syn_types = stype_map


print(f"syn_types proportion: {sum(1 for v in syn_types.values() if v == 'inh') / sum(1 for v in syn_types.values() if v == 'exc') * 100:.2f} %")

syn_types proportion: 30.39 %


'exc'

#### Get Stimulated Neurons

In [5]:
io_neurons = pd.read_csv("./Datasets/Generated/IO_Neurons.csv", index_col="Unnamed: 0")

# Get only input neurons
input_neurons = io_neurons[io_neurons["IO"] == "input"]

# # Get only visual
# input_neurons = input_neurons[input_neurons["additional_annotations"]=="visual"]

input_neurons = list(input_neurons["neuron_id"].astype("string"))

print(f"Input neurons: {len(input_neurons)}")

Input neurons: 480


# Build the Network

In [6]:
soma_params = {
    'L': 20,            # длина
    'diam': 20,         # диаметр
    'Ra': 100,          # аксиальное сопротивление
    'cm': 1,            # емкость мембраны
    
    'gnabar_hh': 0.12,  # проводимость натриевых каналов
    'gkbar_hh': 0.036,  # проводимость калиевых каналов
    'gl_hh': 0.0003,    # утечка (leak) проводимость
    'el_hh': -54.3      # обратный потенциал утечки
}

# Параметры синапсов: отдельно для возбуждающих и подавляющих (NEURON Exp2Syn: e определяет exc/inh)
syn_params_exc = {
    'tau1': 0.5,        # время нарастания (мс)
    'tau2': 2.0,        # время спада (мс)
    'e': 0.0            # обратный потенциал (возбуждающий, ~0 mV)
}
syn_params_inh = {
    'tau1': 0.3,        # время нарастания (GABA быстрее нарастает)
    'tau2': 5.0,        # время спада (типичный decay GABA-A ~4–10 ms)
    'e': -80.0          # обратный потенциал (подавляющий, GABA/Cl- ~ -75..-80 mV)
}
synapse_params = {'exc': syn_params_exc, 'inh': syn_params_inh}

netcon_params = {
    'threshold': 0.0,    # порог (мВ) *это просто чтоб показать работоспособность
    'weight': 0.00059,       # вес синапса (микросименсы)
    'delay': 2.0         # задержка (мс)
}


net = Network(neuron_list)
net.load_graphs(allow_tqdm=True)
net.build_sections(soma_mechanism='hh',
                   soma_params=soma_params,
                   dendrite_mechanism='hh',
                   dendrite_params=neurites_params, 
                  allow_tqdm=True)
net.connect_morphology(allow_tqdm=True)
net.build_synapses(syn_types=syn_types, synapse_params=synapse_params, netcon_params=netcon_params, allow_tqdm=True)

loading graphs:   0%|          | 0/3016 [00:00<?, ?it/s]

building sections:   0%|          | 0/3016 [00:00<?, ?it/s]

connecting morphology:   0%|          | 0/3016 [00:00<?, ?it/s]

building synapses:   0%|          | 0/3016 [00:00<?, ?it/s]

# Run the simulation

In [7]:
# Configure the simulation
net.setup_recording()
net.setup_stimulus(neurons=input_neurons, start_time=100, duration=1000, amplitude=0.2)

# запускаем симуляцию
t, voltages = net.run(duration=2000, dt=0.25)

net.analyze(t, voltages)

net.reset()

# Save and Visualize

In [8]:
exp_name = "3k_inh-exc_70con_2000ms_stim_input_all_amp02_1000ms_000059synweight_2syndelay"
exp_folder = "Sensory_Input_Experiments"

exp_folder_path = f"./Datasets/Generated/Experiments/{exp_folder}"
os.makedirs(exp_folder_path, exist_ok=True)

vis_folder_path = f"./Visualizations/Synaptic_Level/{exp_folder}"
os.makedirs(vis_folder_path, exist_ok=True)

In [9]:
data_pkl = {"t": t, "voltages": voltages}
pkl_path = os.path.join(exp_folder_path, f"{exp_name}.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(data_pkl, f)

In [10]:
out_npz = {"t": np.array(t, dtype=np.float64)}
for nid, v in voltages.items():
    out_npz[f"voltages/{nid}"] = np.array(v, dtype=np.float64)
npz_path = os.path.join(exp_folder_path, f"{exp_name}.npz")
np.savez(npz_path, **out_npz)

In [11]:
save_path = os.path.join(vis_folder_path, f"{exp_name}_2d.png")

plot_results_2d(t, voltages, stim_times=[100, 200], save_path=save_path, interactive=False)


/Data/Drosophila-Larvae-Simulation/scripts/Sim_Visializations.py:145: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
/Data/Drosophila-Larvae-Simulation/scripts/Sim_Visializations.py:145: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/Data/Drosophila-Larvae-Simulation/scripts/Sim_Visializations.py:149: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(save_path, dpi=300)


In [12]:
save_path = os.path.join(vis_folder_path, f"{exp_name}_2d.html")

plot_results_2d(t, voltages, stim_times=[100, 200], save_path=save_path, interactive=True)


In [13]:
save_path = os.path.join(vis_folder_path, f"{exp_name}_3d.png")

plot_results_3d(t, voltages, save_path=save_path, interactive=False)


/Data/Drosophila-Larvae-Simulation/scripts/Sim_Visializations.py:72: UserWarning:

Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.



In [14]:
save_path = os.path.join(vis_folder_path, f"{exp_name}_3d.html")

plot_results_3d(t, voltages, save_path=save_path, interactive=True)
